In [ ]:
pip install torch transformers scikit-learn pandas numpy

In [ ]:
# 1) Load dataset
df = pd.read_csv("/Reviews.csv")

# 2) Map numeric ratings to sentiment classes
def map_score(score):
    if score in [1, 2]:
        return 0  # Negative
    elif score == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

df['label'] = df['Text'].apply(map_score)

# 3) Train-test split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['Text'].tolist(), df['label'].tolist(), test_size=0.2
)

# ✅ Now we can tokenize and continue like before

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
import torch

# 1) Load dataset
df = pd.read_csv("/Reviews.csv")

# 2) Encode string labels to numbers
encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['Score'])
# positive → 2, negative → 0, neutral → 1 (depends on fit order)

# 3) Train-test split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['Score'].tolist(), df['label'].tolist(), test_size=0.2
)

# ✅ Now we can tokenize and continue like before

In [ ]:
# =========================
# CONFIG (edit this only)
# =========================
FILE_PATH = "/Reviews.csv"        # path to CSV
TEXT_COL  = "Text"                    # text column name

# Option A: string labels (uncomment & set)
# LABEL_COL = "Text"               # e.g., 'negative','neutral','positive'
# LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}

# Option B: ratings → sentiment (comment Option A above, uncomment below)
LABEL_COL = None
RATING_COL = "Score"               # e.g., 1–5 stars

MODEL_NAME = "distilbert-base-uncased"      # or "distilbert-base-uncased"
MAX_LEN    = 128
BATCH_SIZE = 16
EPOCHS     = 1
LR         = 2e-5
BALANCE_PER_CLASS = None              # e.g., 10000 for quick balanced subset, or None

# =========================
# CODE (no changes needed)
# =========================
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.optim import AdamW

# 1) Load CSV
df = pd.read_csv(FILE_PATH)
if LABEL_COL:
    needed = [TEXT_COL, LABEL_COL]
else:
    needed = [TEXT_COL, RATING_COL]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

df = df[needed].dropna().drop_duplicates(subset=[TEXT_COL])

# 2) Make numeric labels
if LABEL_COL:
    # String labels -> ints using LABEL_MAP
    unseen = set(df[LABEL_COL].unique()) - set(LABEL_MAP.keys())
    if unseen:
        raise ValueError(f"Unmapped labels found: {unseen}. Update LABEL_MAP.")
    df["label"] = df[LABEL_COL].map(LABEL_MAP)
else:
    # Ratings -> 3-class mapping (edit if you want a different rule)
    def rating_to_label(r):
        r = int(r)
        if r <= 2:  # negative
            return 0
        elif r <= 3:  # neutral
            return 1
        else:  # 8–10 positive
            return 2
    df["label"] = df[RATING_COL].apply(rating_to_label)

# Optional: balance classes for faster/cleaner training
if BALANCE_PER_CLASS is not None:
    df = df.groupby("label", group_keys=False).apply(
        lambda x: x.sample(min(len(x), BALANCE_PER_CLASS), random_state=42)
    )

# 3) Verify classes and split
classes = sorted(df["label"].unique())
num_labels = len(classes)
if num_labels < 2:
    raise ValueError("Need at least 2 classes to train.")
print(f"Classes: {classes} | num_labels={num_labels} | rows={len(df)}")

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df[TEXT_COL].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# 4) Tokenizer + Model (BERT or DistilBERT)
if MODEL_NAME.startswith("distilbert"):
    from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
    tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
    model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
else:
    from transformers import BertTokenizer, BertForSequenceClassification
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

# 5) Dataset/Dataloader
class GenericTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = GenericTextDataset(train_texts, train_labels, tokenizer, max_len=MAX_LEN)
test_dataset  = GenericTextDataset(test_texts,  test_labels,  tokenizer, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

# 6) Train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = AdamW(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            labels=batch["labels"].to(device)
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Avg loss: {total_loss/len(train_loader):.4f}")

# 7) Evaluate
model.eval()
preds, truths = [], []
with torch.no_grad():
    for batch in test_loader:
        logits = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device)
        ).logits
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        truths.extend(batch["labels"].cpu().numpy())

# Label names (if LABEL_MAP provided)
target_names = None
if LABEL_COL:
    inv = {v: k for k, v in LABEL_MAP.items()}
    target_names = [inv[i] for i in classes]

print("\nClassification Report:")
print(classification_report(truths, preds, target_names=target_names))

# 8) Predict helper
def predict_sentiment(text: str):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy().flatten()
        pred = int(np.argmax(probs))
    name = None
    if LABEL_COL:
        inv = {v: k for k, v in LABEL_MAP.items()}
        name = inv.get(pred, str(pred))
    return pred, name, probs

# Quick sanity checks
for t in ["This product is amazing, I love it!",
          "Worst purchase ever. Waste of money.",
          "It's okay, not great but not terrible."]:
    p, name, probs = predict_sentiment(t)
    lab = name if name is not None else p
    print(f"\nText: {t}\nPrediction: {lab} | probs={probs}")